### Environment Helpers

Small post-ingestion environment-population helpers that need to run after
`Canonical_Data` (catalog + schemas exist) and `Document_Data` (PDFs are in
volumes) but are too small or too miscellaneous to warrant their own stage.

**Current helpers**:

- **AI SQL demo on food-safety inspection PDFs** — exercises
  `ai_parse_document`, `ai_classify`, `ai_extract`, and `ai_summarize`
  against a small sample (`PDF_SAMPLE_SIZE` rows) from
  `/Volumes/{CATALOG}/food_safety/reports/` and writes the results to
  `{CATALOG}.food_safety.ai_*_inspections` tables for inspection from the
  SQL editor / a downstream dashboard.
- **ABAC governance + data-quality monitoring** — creates governed
  tags, four UDFs, and four `CREATE POLICY` statements that drive PII
  masking, region-based row filtering, high-value-order gating, and a
  regulated-document deny on `food_safety.ai_extracted_inspections`,
  then turns on schema-level anomaly detection on `food_safety`.  See
  the markdown intro below the AI SQL section for the policy matrix.

Add new helpers in their own cells below.  Keep each idempotent
(`CREATE OR REPLACE`, `CREATE … IF NOT EXISTS`, `MERGE`) and best-effort
(skip gracefully if the inputs it needs are missing) so this stage can be
re-run safely as part of a `bundle run caspers`, and so it can be added to
targets that don't run `Document_Data` without breaking them.

In [ ]:
CATALOG = dbutils.widgets.get("CATALOG")

# Size of the AI SQL demo sample.  `ai_parse_document` bills per page and
# `ai_classify`/`ai_extract`/`ai_summarize` bill per row, so keep this small
# unless you are deliberately running a wider sweep.
PDF_SAMPLE_SIZE = 6
PDF_VOLUME_PATH = f"/Volumes/{CATALOG}/food_safety/reports"

print(f"environment_helpers: catalog={CATALOG}")
print(f"environment_helpers: PDF sample size for AI SQL demo = {PDF_SAMPLE_SIZE}")
print(f"environment_helpers: PDF volume path = {PDF_VOLUME_PATH}")

#### AI SQL demo on inspection PDFs

Reads `PDF_SAMPLE_SIZE` PDFs out of the food-safety `reports` volume and
chains four AI SQL functions over them:

| Function | What we get out | Output table |
|---|---|---|
| `ai_parse_document` | Per-PDF structured `VARIANT` + flattened text | `food_safety.ai_parsed_inspections` |
| `ai_classify` | Bucketed overall risk + dominant violation category | `food_safety.ai_classified_inspections` |
| `ai_extract` | Structured fields (date, location, score, grade) | `food_safety.ai_extracted_inspections` |
| `ai_summarize` | Short plain-English summary per inspection | `food_safety.ai_summarized_inspections` |

All cells are idempotent (`CREATE OR REPLACE`) and gated by the volume
check below so this stage stays safe in targets that don't run
`Document_Data`.

In [ ]:
# Guard: skip the entire AI SQL demo if the food-safety PDF volume is
# absent or empty.  This lets Environment_Helpers be added to targets that
# don't run Document_Data without making the task fail.
try:
    pdf_entries = [
        f for f in dbutils.fs.ls(PDF_VOLUME_PATH)
        if f.name.lower().endswith(".pdf")
    ]
    pdf_count = len(pdf_entries)
except Exception as e:
    print(
        f"⏭️  Cannot access {PDF_VOLUME_PATH} "
        f"({type(e).__name__}: {e}) — skipping AI SQL demo."
    )
    dbutils.notebook.exit("ai_sql_demo: skipped (volume missing)")

if pdf_count == 0:
    print(f"⏭️  No PDFs in {PDF_VOLUME_PATH} — skipping AI SQL demo.")
    dbutils.notebook.exit("ai_sql_demo: skipped (volume empty)")

print(f"✅ Found {pdf_count} PDFs in {PDF_VOLUME_PATH}")
print(f"   Will process the first {min(pdf_count, PDF_SAMPLE_SIZE)} via AI SQL.")

In [ ]:
# ai_parse_document → one row per PDF, with:
#   - the full structured VARIANT (kept for power users)
#   - a flat `text` column that concatenates every text/heading/caption
#     element in document order (the input the downstream AI SQL calls use)
#   - a couple of metadata columns for sanity checks
#
# Cost note: ai_parse_document bills per page, so we LIMIT the source to
# PDF_SAMPLE_SIZE rows.  Remove the LIMIT (and reconsider the volume scan)
# for a production parse pass.
spark.sql(f"""
CREATE OR REPLACE TABLE {CATALOG}.food_safety.ai_parsed_inspections
COMMENT 'Inspection PDFs parsed with ai_parse_document; one row per PDF.'
AS
WITH parsed AS (
  SELECT
    element_at(split(path, '/'), -1)                       AS pdf_name,
    ai_parse_document(content, map('version', '2.0'))      AS doc
  FROM READ_FILES('{PDF_VOLUME_PATH}/', format => 'binaryFile')
  WHERE lower(path) LIKE '%.pdf'
  ORDER BY path
  LIMIT {PDF_SAMPLE_SIZE}
)
SELECT
  pdf_name,
  doc                                                       AS parsed,
  array_join(
    transform(
      filter(
        try_variant_get(
          doc,
          '$.document.elements',
          'ARRAY<STRUCT<type STRING, content STRING>>'
        ),
        e -> e.type IN ('text', 'title', 'section_header', 'caption')
      ),
      e -> e.content
    ),
    '\\n'
  )                                                         AS text,
  size(
    try_variant_get(doc, '$.document.pages', 'ARRAY<STRUCT<id INT>>')
  )                                                         AS num_pages,
  CAST(doc:metadata:file_metadata:file_size AS BIGINT)      AS file_size_bytes
FROM parsed
""")

print(f"✅ Created {CATALOG}.food_safety.ai_parsed_inspections")
display(
    spark.sql(f"""
        SELECT pdf_name, num_pages, file_size_bytes, left(text, 200) AS text_preview
        FROM {CATALOG}.food_safety.ai_parsed_inspections
        ORDER BY pdf_name
    """)
)

In [ ]:
# ai_classify → assign each PDF to one of a fixed label set.  Two passes:
#   - overall_risk: bucketed severity of the inspection as a whole
#   - dominant_category: which violation category dominates the report
#
# Labels are chosen to match the categories the generator actually emits
# (see data/inspections/generate_inspection_reports.py) so the demo is
# interpretable against the ground-truth violations table.
spark.sql(f"""
CREATE OR REPLACE TABLE {CATALOG}.food_safety.ai_classified_inspections
COMMENT 'Per-PDF risk bucket + dominant violation category from ai_classify.'
AS
SELECT
  pdf_name,
  ai_classify(
    text,
    ARRAY(
      'clean_pass',
      'minor_violations_only',
      'major_violations_present',
      'critical_violations_present'
    )
  ) AS overall_risk,
  ai_classify(
    text,
    ARRAY(
      'Temperature Control',
      'Cross-Contamination',
      'Personal Hygiene',
      'Sanitation',
      'Personnel',
      'Facilities',
      'Equipment',
      'Food Labeling',
      'Maintenance',
      'Pest Control',
      'Administrative',
      'Chemical Safety'
    )
  ) AS dominant_violation_category
FROM {CATALOG}.food_safety.ai_parsed_inspections
WHERE text IS NOT NULL
""")

print(f"✅ Created {CATALOG}.food_safety.ai_classified_inspections")
display(
    spark.sql(f"""
        SELECT * FROM {CATALOG}.food_safety.ai_classified_inspections
        ORDER BY pdf_name
    """)
)

In [ ]:
# ai_extract → pull the structured fields the inspection PDFs print on
# their cover page out of the free-text the parser produced.  The result is
# a STRUCT<field STRING, ...> where each label in the array becomes a
# top-level field.
#
# These should track closely with the canonical values in
# {CATALOG}.food_safety.inspections — comparing the two is a quick way to
# eyeball how well the parse-then-extract chain is doing.
spark.sql(f"""
CREATE OR REPLACE TABLE {CATALOG}.food_safety.ai_extracted_inspections
COMMENT 'Structured fields lifted from each inspection PDF via ai_extract.'
AS
SELECT
  pdf_name,
  ai_extract(
    text,
    ARRAY(
      'inspection_date',
      'inspector_name',
      'location_name',
      'address',
      'score',
      'grade',
      'total_violations'
    )
  ) AS extracted
FROM {CATALOG}.food_safety.ai_parsed_inspections
WHERE text IS NOT NULL
""")

print(f"✅ Created {CATALOG}.food_safety.ai_extracted_inspections")
display(
    spark.sql(f"""
        SELECT
          pdf_name,
          extracted.inspection_date,
          extracted.inspector_name,
          extracted.location_name,
          extracted.score,
          extracted.grade,
          extracted.total_violations
        FROM {CATALOG}.food_safety.ai_extracted_inspections
        ORDER BY pdf_name
    """)
)

In [ ]:
# ai_summarize → a short plain-English summary per PDF.  The second arg
# is the max-words target (the function treats it as a soft bound).
#
# Useful as a "what's in this report" tooltip in dashboards or as the
# first column a reviewer sees when triaging a stack of inspections.
spark.sql(f"""
CREATE OR REPLACE TABLE {CATALOG}.food_safety.ai_summarized_inspections
COMMENT 'Short plain-English summary of each inspection PDF via ai_summarize.'
AS
SELECT
  pdf_name,
  ai_summarize(text, 60) AS summary
FROM {CATALOG}.food_safety.ai_parsed_inspections
WHERE text IS NOT NULL
""")

print(f"✅ Created {CATALOG}.food_safety.ai_summarized_inspections")
display(
    spark.sql(f"""
        SELECT * FROM {CATALOG}.food_safety.ai_summarized_inspections
        ORDER BY pdf_name
    """)
)

print()
print("✅ AI SQL helper complete — tables written to "
      f"{CATALOG}.food_safety.ai_*_inspections")

#### ABAC governance + Data Quality monitoring (DAIS 2026 beat)

Sets up an Attribute-Based Access Control story across the Casper's data
estate and turns on schema-level anomaly detection so the catalog has a
data-quality signal alongside the ABAC policies. Each cell is best-effort
and idempotent — missing securables, older runtimes (ABAC needs DBR 16.4+),
older SDKs (Data Quality monitoring needs the new `w.data_quality` service),
or missing demo groups will print a warning and continue, never fail the
stage. The ABAC story is:

| # | Policy | Where | Effect |
|---|---|---|---|
| 1 | PII column mask | `lakeflow.silver_order_items.customer_addr` | Replaces address with `***` unless caller is in `caspers_pii_readers` |
| 2 | Region row filter | `simulator.locations` | Drops rows whose location-code's region the caller can't see |
| 3 | High-value order gate | `lakeflow.gold_order_header.order_revenue` | Hides revenue > $500 unless caller is `caspers_finance` / `caspers_managers` |
| 4 | Regulated documents | `food_safety.ai_extracted_inspections` | Hides all rows unless caller is `caspers_compliance` |

The demo groups (`caspers_pii_readers`, `caspers_us_users`,
`caspers_emea_users`, `caspers_geo_admins`, `caspers_finance`,
`caspers_managers`, `caspers_compliance`) don't need to exist for these
policies to be created — `is_account_group_member` returns false for
non-existent groups, which means the masking/filtering simply applies to
*everyone* (workspace admins still bypass). Create the groups in the
Account console if you want to demonstrate the unmasked perspective from
a separate identity. See `demos/dais2026-runbooks/SETUP.ipynb`.

In [ ]:
# ABAC step 1/4: governance schema + governed tags + tag application.
#
# Tags namespaced under `caspers.*` so they don't collide with workspace-wide
# tag policies the customer may already have. Tagging a non-existent column
# fails the ALTER, so each apply is wrapped in try/except. SET TAGS is
# idempotent — re-running the cell is a no-op.

GOV_SCHEMA = f"{CATALOG}._security"

# Per-securable tags we want to exist before the ABAC policies reference them.
GOV_TAGS = ["caspers.pii", "caspers.geo_region", "caspers.value_class", "caspers.sensitivity"]

# (full_object_name, kind, tag_key=value_pairs)  kind ∈ {"table_column", "table"}
TAG_TARGETS = [
    # PII on the customer address baked into each silver order item row.
    (f"{CATALOG}.lakeflow.silver_order_items",  "table_column",  "customer_addr",  {"caspers.pii": "address"}),
    # Region marker on the location code (sfo/nyc/chi/lax = US; lon/muc/ams/via = EMEA).
    (f"{CATALOG}.simulator.locations",          "table_column",  "location_code",  {"caspers.geo_region": "true"}),
    # Per-order revenue — the column we'll gate on for the high-value policy.
    (f"{CATALOG}.lakeflow.gold_order_header",   "table_column",  "order_revenue",  {"caspers.value_class": "high_value"}),
    # Table-level sensitivity tag — entire AI-extracted inspections table is regulated.
    (f"{CATALOG}.food_safety.ai_extracted_inspections", "table", None,             {"caspers.sensitivity": "restricted"}),
]

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {GOV_SCHEMA} COMMENT 'Casper\\'s ABAC UDFs and policy artifacts.'")
print(f"\u2705 Schema ready: {GOV_SCHEMA}")

# Create governed tags at the metastore level.  Requires tag-policy admin.
# Best-effort: tag may already exist (idempotent), or the caller may not have
# the privilege, in which case ALTER TABLE ... SET TAGS still works for
# non-governed tags as a graceful fallback.
for tag in GOV_TAGS:
    try:
        spark.sql(f"CREATE TAG IF NOT EXISTS `{tag}`")
        print(f"  \u2705 governed tag: {tag}")
    except Exception as e:
        print(f"  \u26a0\ufe0f  could not create governed tag {tag} ({type(e).__name__}); falling back to plain table tags. Detail: {str(e).splitlines()[0]}")

# Apply tags.  Wrap each in try/except so a missing table (target not yet
# materialised by Lakeflow, or skipped in a smaller target) just prints.
for full_name, kind, col, tags in TAG_TARGETS:
    tag_clause = ", ".join(f"'{k}' = '{v}'" for k, v in tags.items())
    if kind == "table":
        sql = f"ALTER TABLE {full_name} SET TAGS ({tag_clause})"
    else:
        sql = f"ALTER TABLE {full_name} ALTER COLUMN `{col}` SET TAGS ({tag_clause})"
    try:
        spark.sql(sql)
        print(f"  \u2705 tagged: {full_name}{f'.{col}' if col else ''} \u2192 {tag_clause}")
    except Exception as e:
        print(f"  \u26a0\ufe0f  skipped {full_name}{f'.{col}' if col else ''}: {str(e).splitlines()[0]}")

In [ ]:
# ABAC step 2/4: UDFs that implement the row-filter / column-mask logic.
#
# These are tiny SQL functions that each ABAC policy references. Keeping the
# logic in named UDFs makes the policies declarative and lets the security
# team audit/edit the rules independently of where they're applied.
# All four are `CREATE OR REPLACE` so re-running the cell updates the body
# in place without breaking the policies that reference them.

UDFS = [
    # 1. PII column mask.  Returns the original address only to members of
    # `caspers_pii_readers`; everyone else sees `***` (workspace admins
    # always bypass row filters / column masks regardless of this UDF).
    f"""
    CREATE OR REPLACE FUNCTION {GOV_SCHEMA}.mask_pii(value STRING)
    RETURNS STRING
    COMMENT 'Mask PII unless caller is in caspers_pii_readers group.'
    RETURN CASE
      WHEN is_account_group_member('caspers_pii_readers') THEN value
      ELSE '***'
    END
    """,

    # 2. Region row filter.  Returns TRUE (row visible) if the caller is in
    # `caspers_geo_admins` OR in the region group matching the row's
    # location_code prefix. US codes: sfo/nyc/chi/lax; EMEA: lon/muc/ams/via.
    f"""
    CREATE OR REPLACE FUNCTION {GOV_SCHEMA}.filter_by_region(location_code STRING)
    RETURNS BOOLEAN
    COMMENT 'Region-based row filter: US codes for caspers_us_users, EMEA codes for caspers_emea_users, all for caspers_geo_admins.'
    RETURN
      is_account_group_member('caspers_geo_admins')
      OR (location_code IN ('sfo','nyc','chi','lax') AND is_account_group_member('caspers_us_users'))
      OR (location_code IN ('lon','muc','ams','via') AND is_account_group_member('caspers_emea_users'))
    """,

    # 3. High-value order row filter.  Orders <= $500 are visible to anyone;
    # higher-revenue orders only to caspers_finance / caspers_managers.
    f"""
    CREATE OR REPLACE FUNCTION {GOV_SCHEMA}.filter_high_value(order_revenue DOUBLE)
    RETURNS BOOLEAN
    COMMENT 'Hide orders > $500 unless caller is in caspers_finance or caspers_managers.'
    RETURN
      COALESCE(order_revenue, 0) <= 500
      OR is_account_group_member('caspers_finance')
      OR is_account_group_member('caspers_managers')
    """,

    # 4. Regulated-document row filter.  Tables tagged `caspers.sensitivity =
    # restricted` show no rows unless caller is in caspers_compliance.  This
    # is the "table-level deny" pattern — no per-row data needed, so the
    # function takes no arguments (the policy uses WHEN-clause matching).
    f"""
    CREATE OR REPLACE FUNCTION {GOV_SCHEMA}.filter_regulated_docs()
    RETURNS BOOLEAN
    COMMENT 'Hide all rows of tables tagged sensitivity=restricted unless caller is in caspers_compliance.'
    RETURN is_account_group_member('caspers_compliance')
    """,
]

for udf_sql in UDFS:
    try:
        spark.sql(udf_sql)
        # Pull function name from the SQL for the print
        name = udf_sql.split("FUNCTION", 1)[1].split("(", 1)[0].strip()
        print(f"\u2705 created/updated UDF: {name}")
    except Exception as e:
        print(f"\u26a0\ufe0f  UDF create failed: {str(e).splitlines()[0]}")

# Grant EXECUTE on each UDF to `account users` so the ABAC policies can
# actually invoke them in non-admin sessions.
for fname in ["mask_pii", "filter_by_region", "filter_high_value", "filter_regulated_docs"]:
    try:
        spark.sql(f"GRANT EXECUTE ON FUNCTION {GOV_SCHEMA}.{fname} TO `account users`")
    except Exception as e:
        print(f"\u26a0\ufe0f  GRANT EXECUTE on {fname} failed: {str(e).splitlines()[0]}")

In [ ]:
# ABAC step 3/4: declare the 4 policies.
#
# Each policy is attached at the schema (or table) level and uses MATCH
# COLUMNS / WHEN to bind itself to the right securables via the governed
# tags we set in step 1.  CREATE OR REPLACE makes the cell idempotent.
# Requires Databricks Runtime 16.4+ (CREATE POLICY syntax) on the SQL
# warehouse used by readers.

POLICIES = [
    # 1. PII column mask over any column tagged `caspers.pii=*` in the
    # lakeflow schema — automatically picks up customer_addr (and any
    # future PII-tagged columns).
    (f"{CATALOG}.lakeflow",
     "SCHEMA",
     "caspers_mask_pii",
     f"""
     CREATE OR REPLACE POLICY caspers_mask_pii
     ON SCHEMA {CATALOG}.lakeflow
     COMMENT 'Mask PII columns unless caller is caspers_pii_readers.'
     COLUMN MASK {GOV_SCHEMA}.mask_pii
     TO `account users`
     EXCEPT `caspers_pii_readers`
     FOR TABLES
     MATCH COLUMNS has_tag('caspers.pii') AS pii_col
     ON COLUMN pii_col
     """),

    # 2. Region row filter on any table whose location_code column carries
    # the `caspers.geo_region` tag (currently just simulator.locations).
    (f"{CATALOG}.simulator",
     "SCHEMA",
     "caspers_region_filter",
     f"""
     CREATE OR REPLACE POLICY caspers_region_filter
     ON SCHEMA {CATALOG}.simulator
     COMMENT 'Show only rows whose location_code is in the caller\\'s allowed region.'
     ROW FILTER {GOV_SCHEMA}.filter_by_region
     TO `account users`
     EXCEPT `caspers_geo_admins`
     FOR TABLES
     MATCH COLUMNS has_tag('caspers.geo_region') AS region_col
     USING COLUMNS (region_col)
     """),

    # 3. High-value-order gate on tables with a column tagged
    # `caspers.value_class = high_value` (currently gold_order_header).
    (f"{CATALOG}.lakeflow",
     "SCHEMA",
     "caspers_high_value_gate",
     f"""
     CREATE OR REPLACE POLICY caspers_high_value_gate
     ON SCHEMA {CATALOG}.lakeflow
     COMMENT 'Hide orders > $500 unless caller is caspers_finance or caspers_managers.'
     ROW FILTER {GOV_SCHEMA}.filter_high_value
     TO `account users`
     EXCEPT `caspers_finance`, `caspers_managers`
     FOR TABLES
     MATCH COLUMNS has_tag_value('caspers.value_class', 'high_value') AS revenue_col
     USING COLUMNS (revenue_col)
     """),

    # 4. Regulated-doc deny on any table tagged
    # `caspers.sensitivity = restricted` in food_safety.
    (f"{CATALOG}.food_safety",
     "SCHEMA",
     "caspers_regulated_docs",
     f"""
     CREATE OR REPLACE POLICY caspers_regulated_docs
     ON SCHEMA {CATALOG}.food_safety
     COMMENT 'Hide every row of tables tagged sensitivity=restricted unless caller is caspers_compliance.'
     ROW FILTER {GOV_SCHEMA}.filter_regulated_docs
     TO `account users`
     EXCEPT `caspers_compliance`
     FOR TABLES
     WHEN has_tag_value('caspers.sensitivity', 'restricted')
     """),
]

for securable, kind, name, sql in POLICIES:
    try:
        spark.sql(sql)
        print(f"\u2705 created policy {name} on {kind} {securable}")
    except Exception as e:
        msg = str(e).splitlines()[0]
        if "16.4" in msg or "POLICY" in msg.upper() or "syntax" in msg.lower():
            print(f"\u26a0\ufe0f  policy {name} skipped \u2014 needs DBR 16.4+ on the editor warehouse: {msg}")
        else:
            print(f"\u26a0\ufe0f  policy {name} failed: {msg}")

In [ ]:
# ABAC step 4/4: turn on schema-level anomaly detection on food_safety.
#
# Uses the new `w.data_quality` service (replaces the deprecated
# `w.quality_monitors`).  Anomaly detection runs continuously across every
# table in the schema, surfacing completeness / freshness regressions in
# Catalog Explorer.  The DBR 16+/SDK 0.83+ contract:
#
#   w.data_quality.create_monitor(
#       Monitor(
#           object_type="schema",
#           object_id=schema.schema_id,
#           anomaly_detection_config=AnomalyDetectionConfig(),
#       )
#   )
#
# Requires:
#   - Workspace preview flag: "Data quality monitoring with anomaly
#     detection (workspace level)" enabled (Admin Console > Previews).
#   - Serverless compute enabled (already a Casper's prereq).
#   - databricks-sdk >= 0.83 (has `w.data_quality`).
#   - MANAGE on the parent catalog OR MANAGE on the schema itself.

from databricks.sdk import WorkspaceClient

w = WorkspaceClient()
TARGET_SCHEMA = f"{CATALOG}.food_safety"

try:
    from databricks.sdk.service.dataquality import Monitor, AnomalyDetectionConfig

    if not hasattr(w, "data_quality"):
        raise RuntimeError("databricks-sdk too old (need >= 0.83 for w.data_quality)")

    schema = w.schemas.get(full_name=TARGET_SCHEMA)
    monitor_kwargs = dict(
        object_type="schema",
        object_id=schema.schema_id,
        anomaly_detection_config=AnomalyDetectionConfig(),
    )

    # Re-running is harmless: get_monitor 404s when none exists, in which
    # case we create one; otherwise we leave the existing monitor alone.
    try:
        existing = w.data_quality.get_monitor(object_type="schema", object_id=schema.schema_id)
        print(f"\u267b\ufe0f  data-quality monitor already exists on {TARGET_SCHEMA}")
    except Exception:
        w.data_quality.create_monitor(monitor=Monitor(**monitor_kwargs))
        print(f"\u2705 enabled anomaly detection on {TARGET_SCHEMA}")
        print(f"   \u2192 view results in Catalog Explorer \u2192 {TARGET_SCHEMA} \u2192 Data Quality")
except Exception as e:
    msg = str(e).splitlines()[0]
    print(f"\u26a0\ufe0f  data-quality monitor on {TARGET_SCHEMA} skipped: {msg}")
    print("   Common causes: workspace preview flag not on, SDK < 0.83, "
          "missing MANAGE privilege, or food_safety schema absent for this target.")

print()
print("\u2705 Environment_Helpers complete \u2014 ABAC tags/UDFs/policies and "
      "data-quality monitor are in place where supported.")